In [0]:
target_catalog = "demo_db"
target_schema = "demo"
source_nm = "customer"

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
customer_df = spark.read.csv("/Volumes/demo_db/demo/data_csv_volume/customer*.csv",inferSchema = True,header = True)




In [0]:
customer_df.display()

CREATE Delata Table using DataFramewriter API

In [0]:
# Careating delta Table
customer_df.write.mode("append").format("delta").saveAsTable(f"{target_catalog}.{target_schema}.{source_nm}")

# Checking data in delta table
spark.sql(f"select * from {target_catalog}.{target_schema}.{source_nm}").display()

Create delta table using CREATE DML statement

In [0]:
spark.sql(f"USE CATALOG {target_catalog}")
spark.sql(f"USE SCHEMA {target_schema}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {target_catalog}.{target_schema}.customer_transaction
(
    customer_id INT,
    name STRING,
    dob DATE,
    gender STRING,
    city STRING,
    join_date DATE,
    status STRING,
    email STRING,
    phone_number STRING,
    preferred_channel STRING,
    occupation STRING,
    income_range STRING,
    risk_segment STRING
)
using delta
""")

In [0]:
customer_df.write.mode("overwrite").saveAsTable(f"{target_catalog}.{target_schema}.customer_transaction")

# Checking data into the table
spark.sql(f"select * from {target_catalog}.{target_schema}.customer_transaction").display()
spark.sql(f"select count(*) from {target_catalog}.{target_schema}.customer_transaction").display()


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS delta.`/Volumes/demo_db/demo/data_csv_volume/customer/`
(
    customer_id INT,
    name STRING,
    dob DATE,
    gender STRING,
    city STRING,
    join_date DATE,
    status STRING,
    email STRING,
    phone_number STRING,
    preferred_channel STRING,
    occupation STRING,
    income_range STRING,
    risk_segment STRING
)
using delta
""")

In [0]:
customer_df.write.mode("append").save(f"/Volumes/demo_db/demo/data_csv_volume/customer/")
spark.sql(f"select * from delta.`/Volumes/demo_db/demo/data_csv_volume/customer/`").display()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DateType,DatetimeType
DeltaTable.createIfNotExists(spark)\
    .tableName(f"{target_catalog}.{target_schema}.customer_delta")\
        .addColumns(
            [
                StructField("customer_id",IntegerType()),
                StructField("name", StringType(),True),
                StructField("dob", DateType(),True),
                StructField("gender", StringType(),True),
                StructField( "city", StringType(), True),
                StructField( "join_date", DateType(), True),
                StructField( "status", StringType(), True),
                StructField( "email", StringType(), True),
                StructField( "phone_number", StringType(), True),
                StructField( "preferred_channel", StringType(), True),
                StructField( "occupation", StringType(), True),
                StructField( "income_range", StringType(), True),
                StructField( "risk_segment" , StringType(), True)
            ]
        ).execute()

In [0]:
customer_df.write.mode("overwrite").saveAsTable(f"{target_catalog}.{target_schema}.customer_delta")
spark.sql(f"select * from {target_catalog}.{target_schema}.customer_delta").display()